# Deep Q-Learning (DQN) on CartPole

This notebook moves from tabular reinforcement learning to **Deep Q-Learning (DQN)**. In FrozenLake, a Q-table stored one value for every state-action pair. CartPole has continuous observations, so we replace the table with a neural network:

$$Q(s,a) \quad \longrightarrow \quad Q(s,a;\theta).$$

The rest of the idea remains familiar: use an $\varepsilon$-greedy behaviour policy, observe transitions, and learn from a Bellman target. DQN adds two ingredients that make neural-network training more stable:

1. an **experience-replay buffer**, which lets us train on randomly sampled past transitions;
2. a separate **target network**, which supplies a slowly changing learning target.

In [ ]:
import gymnasium as gym
import numpy as np
import random
import time
from collections import deque

import torch
import torch.nn as nn
from matplotlib import pyplot as plt

## Configuration

In [ ]:
# Training configuration
NUM_EPISODES = 500 
MAX_STEPS_PER_EPISODE = 500
GAMMA = 0.9
LEARNING_RATE = 0.001
BATCH_SIZE = 64
BUFFER_SIZE = 50000
MIN_BUFFER = 1000
TARGET_UPDATE_FREQ = 10
HIDDEN_SIZE = 128

# Epsilon-greedy exploration schedule
EPSILON_MAX = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.01

SEED = 42

## Environment

`CartPole-v1` returns four continuous observations:

1. cart position;
2. cart velocity;
3. pole angle;
4. pole angular velocity.

There are two actions: `0` pushes the cart left and `1` pushes it right. The agent receives reward $+1$ at every step. An episode terminates if the pole tilts too far or the cart moves too far from the centre. It is truncated after 500 steps; Gymnasium considers an average return of 475 over 100 episodes to be solved.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = gym.make(
    "CartPole-v1",
    max_episode_steps=MAX_STEPS_PER_EPISODE,
)
env.action_space.seed(SEED)
env.observation_space.seed(SEED)

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("State size:", state_size)
print("Action size:", action_size)
print("Maximum episode steps:", env.spec.max_episode_steps)
print("Reward threshold:", env.spec.reward_threshold)
print("Device:", device)

## Q-network

The network receives the four observations and returns two action-values:

$$[x, \dot{x}, \theta, \dot{\theta}]\;\longrightarrow\;[Q(s,\mathrm{left}), Q(s,\mathrm{right})].$$

Unlike FrozenLake, no state encoding or discretisation is needed: the observation is already a vector of floating-point values.

In [ ]:
class QNetwork(nn.Module):
    """Map a CartPole observation to one Q-value per action."""

    def __init__(self, state_size, action_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size),
        )

    def forward(self, state):
        return self.net(state)

In [ ]:
# Online network: updated by gradient descent
q_net = QNetwork(state_size, action_size, HIDDEN_SIZE).to(device)

# Target network: supplies stable Bellman targets
target_net = QNetwork(state_size, action_size, HIDDEN_SIZE).to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = torch.optim.Adam(q_net.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss() #nn.SmoothL1Loss() # check this term
replay_buffer = deque(maxlen=BUFFER_SIZE)

## Replay buffer and policy helper

The replay buffer stores transitions $(s,a,r,s',\mathrm{terminated})$. The behaviour policy explores with probability $\varepsilon$ and otherwise chooses the action with the largest predicted Q-value.

In [ ]:
def epsilon_greedy(state, epsilon):
    """Choose a random action or the action with the largest Q-value."""
    if random.random() < epsilon:
        return env.action_space.sample()

    state_tensor = torch.as_tensor(
        state, dtype=torch.float32, device=device
    ).unsqueeze(0)

    with torch.no_grad():
        q_values = q_net(state_tensor)

    return int(q_values.argmax(dim=1).item())

## Learning step

For each sampled transition, DQN fits the online network to the target

$$y = r + \gamma(1-\mathrm{terminated})\max_{a'}Q(s',a';\theta^-).$$

`terminated` is true when the pole falls or the cart leaves the allowed region. `truncated` only means that the 500-step time limit was reached, so it does not remove the bootstrap term. The target is computed without gradients because only the online prediction $Q(s,a;\theta)$ is being fitted.

In [ ]:
def train_step():
    """Perform one DQN update from a random replay minibatch."""
    if len(replay_buffer) < MIN_BUFFER:
        return None

    batch = random.sample(replay_buffer, BATCH_SIZE)
    states, actions, rewards, next_states, terminated = zip(*batch)

    states = torch.as_tensor(
        np.array(states), dtype=torch.float32, device=device
    )
    actions = torch.as_tensor(
        actions, dtype=torch.int64, device=device
    ).unsqueeze(1)
    rewards = torch.as_tensor(
        rewards, dtype=torch.float32, device=device
    ).unsqueeze(1)
    next_states = torch.as_tensor(
        np.array(next_states), dtype=torch.float32, device=device
    )
    terminated = torch.as_tensor(
        terminated, dtype=torch.float32, device=device
    ).unsqueeze(1)

    all_q_values = q_net(states)
    selected_q_values = all_q_values.gather(1, actions)

    with torch.no_grad():
        all_next_q_values = target_net(next_states)
        max_next_q_values = all_next_q_values.max(dim=1, keepdim=True)[0]
        targets = rewards + GAMMA * max_next_q_values * (1.0 - terminated)

    loss = loss_fn(selected_q_values, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

### Check the tensor shapes

A minibatch contains 64 observations. The network returns two Q-values per observation, while the chosen Q-values and Bellman targets each contain one value per transition.

In [ ]:
example_states = torch.zeros((BATCH_SIZE, state_size), device=device)
example_actions = torch.zeros((BATCH_SIZE, 1), dtype=torch.int64, device=device)
example_q_values = q_net(example_states)
example_selected_q_values = example_q_values.gather(1, example_actions)
example_targets = torch.zeros((BATCH_SIZE, 1), device=device)

print("Observations:     ", tuple(example_states.shape))
print("All Q-values:     ", tuple(example_q_values.shape))
print("Selected Q-values:", tuple(example_selected_q_values.shape))
print("Targets:          ", tuple(example_targets.shape))

## Training

For every environment step we store one transition and, once the replay buffer is warm, perform one gradient update. Every 10 episodes the target network is replaced by a copy of the online network.

In [ ]:
rewards_all_episodes = []
steps_all_episodes = []
losses = []
epsilon = EPSILON_MAX

start_time = time.time()

for episode in range(NUM_EPISODES):
    if episode == 0:
        state, info = env.reset(seed=SEED)
    else:
        state, info = env.reset()

    episode_reward = 0.0
    steps_taken = 0

    for step in range(MAX_STEPS_PER_EPISODE):
        action = epsilon_greedy(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)

        # Store copies because NumPy arrays can otherwise share memory.
        replay_buffer.append((
            state.copy(),
            action,
            reward,
            next_state.copy(),
            float(terminated),
        ))

        loss = train_step()
        if loss is not None:
            losses.append(loss)

        state = next_state
        episode_reward += reward
        steps_taken = step + 1

        if terminated or truncated:
            break

    rewards_all_episodes.append(episode_reward)
    steps_all_episodes.append(steps_taken)

    epsilon = EPSILON_MIN + (EPSILON_MAX - EPSILON_MIN) * np.exp(
        -EPSILON_DECAY * (episode + 1)
    )

    if (episode + 1) % TARGET_UPDATE_FREQ == 0:
        target_net.load_state_dict(q_net.state_dict())

    if episode == 0 or (episode + 1) % 50 == 0:
        recent_mean = np.mean(rewards_all_episodes[-20:])
        print(
            f"Episode {episode + 1:3d}: "
            f"reward={episode_reward:5.0f}, "
            f"mean20={recent_mean:6.1f}, "
            f"epsilon={epsilon:.3f}"
        )

training_minutes = (time.time() - start_time) / 60
print(f"Training time: {training_minutes:.2f} minutes")
print(f"Number of elements in replay buffer: {len(replay_buffer)}")
print(f"NN training updates: {len(losses)}")

## Learning curves

CartPole's reward equals the number of steps survived, so the return and episode length contain the same values. They are shown separately to match the earlier tabular notebooks and to emphasise their different meanings.

In [ ]:
window = 20
rolling_rewards = np.convolve(
    np.array(rewards_all_episodes),
    np.ones(window) / window,
    mode="valid",
)

plt.figure(figsize=(8, 5))
plt.plot(np.arange(window, NUM_EPISODES + 1), rolling_rewards)
plt.axhline(475, color="tab:red", linestyle="--", label="Solved threshold")
plt.title(f"Rolling Average Return (window = {window} episodes)")
plt.xlabel("Episode")
plt.ylabel("Average return")
plt.ylim(0, MAX_STEPS_PER_EPISODE + 10)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps_all_episodes, alpha=0.65)
plt.title("Number of Steps per Episode")
plt.xlabel("Episode")
plt.ylabel("Steps")
plt.ylim(0, MAX_STEPS_PER_EPISODE + 10)
plt.grid(True)
plt.tight_layout()
plt.show()

## Evaluate the greedy policy

Training uses exploration, so its returns do not directly measure the final policy. We now set $\varepsilon=0$ and evaluate the greedy policy over 100 fresh episodes.

In [ ]:
q_net.eval()
evaluation_returns = []

for evaluation_episode in range(100):
    state, info = env.reset(seed=SEED + 1000 + evaluation_episode)
    episode_return = 0.0

    for step in range(MAX_STEPS_PER_EPISODE):
        action = epsilon_greedy(state, epsilon=0.0)
        state, reward, terminated, truncated, info = env.step(action)
        episode_return += reward

        if terminated or truncated:
            break

    evaluation_returns.append(episode_return)

evaluation_returns = np.array(evaluation_returns)
mean_return = evaluation_returns.mean()

print(f"Mean return:     {mean_return:.2f}")
print(f"Standard dev.:   {evaluation_returns.std():.2f}")
print(f"Minimum return:  {evaluation_returns.min():.0f}")
print(f"Maximum return:  {evaluation_returns.max():.0f}")
print(f"Solved (>= 475): {mean_return >= 475}")

## One greedy rollout

Instead of opening a separate rendering window, record one greedy episode and plot two important state variables. A successful policy keeps both the cart position and pole angle close to zero.

In [ ]:
state, info = env.reset(seed=SEED + 2000)
rollout_states = [state.copy()]
rollout_return = 0.0

for step in range(MAX_STEPS_PER_EPISODE):
    action = epsilon_greedy(state, epsilon=0.0)
    state, reward, terminated, truncated, info = env.step(action)
    rollout_states.append(state.copy())
    rollout_return += reward

    if terminated or truncated:
        break

rollout_states = np.array(rollout_states)
time_steps = np.arange(len(rollout_states))

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].plot(time_steps, rollout_states[:, 0])
axes[0].axhline(-2.4, color="red", linestyle="--", label="Position limits")
axes[0].axhline(2.4, color="red", linestyle="--")
axes[0].set_ylabel("Cart position")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(time_steps, np.degrees(rollout_states[:, 2]))
axes[1].axhline(-12.0, color="red", linestyle="--", label="Pole angle limits")
axes[1].axhline(12.0, color="red", linestyle="--")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Pole angle (degrees)")
axes[1].grid(True)
axes[1].legend()

fig.suptitle(f"Greedy Rollout (return = {rollout_return:.0f})")
fig.tight_layout()
plt.show()

print(f"Rollout return: {rollout_return:.0f}")
print(f"Rollout steps:  {len(rollout_states) - 1}")

## Save the trained network

A Q-table was saved as a NumPy array in the tabular notebooks. Here the learned Q-function is stored as the PyTorch network's state dictionary. The same `QNetwork` architecture must be created before these weights are loaded later.

In [ ]:
MODEL_PATH = "dqn_cartpole_policy.pth"
torch.save(q_net.state_dict(), MODEL_PATH)

# Check that the saved state dictionary can be loaded.
saved_weights = torch.load(MODEL_PATH, map_location=device, weights_only=True)
check_net = QNetwork(state_size, action_size, HIDDEN_SIZE).to(device)
check_net.load_state_dict(saved_weights)
check_net.eval()

print("Saved and reloaded:", MODEL_PATH)
env.close()